<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/credit-underwriting/lessons/P05-L01-pd-vintages-and-macro/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/credit-underwriting/lessons/P05-L01-pd-vintages-and-macro/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/credit-underwriting/lessons/P05-L01-pd-vintages-and-macro/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/credit-underwriting/lessons/P05-L01-pd-vintages-and-macro/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P05-L01 · PD modelling with vintages and macro overlays

**You will build:** the vintage curves of a consumer loan book with the censoring handled
correctly, a discrete-time hazard model for the probability of default (PD) by month on
book, fitted by Newton's method that you write and check yourself, and a macro overlay that
turns the fitted model into PD term structures under two economic scenarios. Then you will
back-test it out of time, by vintage, through a recession the model was never shown, and
measure how much a random split, a peek at the future and a leaky feature each flatter that
back-test.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** `T00-L01-the-8gb-track` (what a declared compute budget means). Python
and numpy indexing. No credit or accounting background is assumed: each term is defined in a
sentence where it first appears. No pandas and no scikit-learn: every estimator here is yours.

By the end you will be able to:

1. Implement a risk table and vintage default curves in which a censored loan is neither a
   default nor a good loan, and a month that was never observed stays unobserved.
2. Implement a discrete-time hazard model, one intercept per month on book plus a macro
   covariate, fitted with Newton's method and checked against a numerical gradient.
3. Implement PD term structures under macro scenarios, and the 12-month and lifetime expected
   credit losses that follow from them.
4. Measure out-of-time calibration by vintage, and how much a random split, future macro in
   the training window and an outcome-derived feature each flatter a back-test.
5. Measure a point-in-time and a through-the-cycle PD from one fitted model, and say which
   question each one answers.

A note on the data before anything else: **it is synthetic, and the generator is in this
notebook.** Nothing is downloaded. Real loan-level data of the right kind sit behind
registration walls or non-commercial licences, and the point of a planted book is that you
can check your method against the truth you can see. `meta.yaml` declares it synthetic.
What you conclude here about *method* transfers; what you conclude about *these numbers*
describes no real lender.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`

**This lesson assumes you can already:**

- write numpy code on 2-D arrays: row and column indexing, boolean masks, cumulative sums and cumulative products
- read a logistic model: turn a log-odds value into a probability with the sigmoid and say what a coefficient does to the odds
- chain per-period chances into the probability of lasting several periods, as a running product of one minus the per-period default chance
- solve a small linear system A x = b with numpy.linalg.solve

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Generalized Linear Models, chapter 7: Survival Models](https://grodri.github.io/glms/notes/c7.pdf) · Germán Rodríguez (Princeton) · *free* — section 7.2 on censoring and section 7.6 on discrete-time models, where the hazard becomes a logistic regression with one parameter per period: the model this lesson fits
- [Generalized Linear Models, chapter 3: Logit Models for Binary Data](https://grodri.github.io/glms/notes/c3.pdf) · Germán Rodríguez (Princeton) · *free* — section 3.2.1 gives the binomial log-likelihood and the Fisher scoring procedure (iteratively re-weighted least squares) behind the loss and fitting loop
- [CS231n: Neural Networks Part 3, section Gradient checks](https://cs231n.github.io/neural-networks-3/) · Stanford CS231n (Karpathy et al.) · *free* — why a centred difference beats a one-sided one, and how to compare an analytic gradient with a numerical one by relative error, as in the gradient-check exercise
- [Frequently Asked Questions on the New Accounting Standard on Financial Instruments, Credit Losses](https://www.federalreserve.gov/supervisionreg/srletters/sr1908a1.pdf) · Federal Reserve, FDIC, NCUA and OCC · *free* — plain-language answers on lifetime expected credit loss, reasonable and supportable forecasts and vintage disclosure, the vocabulary behind the PD term structure and the loss exercise
- [Credit Risk Analytics: Measurement Techniques, Applications, and Examples in SAS](https://www.wiley.com/en-us/Credit+Risk+Analytics%3A+Measurement+Techniques%2C+Applications%2C+and+Examples+in+SAS-p-9781119143987) · Baesens, Roesch and Scheule (Wiley) · *paid* — chapter 6, Probabilities of Default (PD): Discrete-Time Hazard Models, the model this lesson fits, then chapter 13 on model validation and chapter 14 on stress testing; examples in SAS

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import os
import signal
import sys
import threading
import time
import traceback
from typing import Any, Callable, NamedTuple

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

# The book. 48 monthly origination cohorts ("vintages"), 1,800 loans each, every loan a
# 36-month instalment loan. Months are counted from 0; vintage v is the month v loans were
# made, and a loan of age a is in calendar month v + a. Data stop at month T_END, so recent
# vintages are only part-way through their lives: that is the whole difficulty.
N_VINTAGES = 48
LOANS_PER_VINTAGE = 1800
MAX_AGE = 36
T_END = 65
SEED = 20260929

# The macro variable is an unemployment-like rate in percent. Models see it as `stress`, the
# points it sits above this reference level, so that 0.0 means "an ordinary month".
U_REF = 5.0

# The back-test protocol. A model is "built" at calendar month C_CUT from vintages 0..V_OOT-1
# and must then predict the vintages from V_OOT onwards, which it has never seen.
V_OOT = 30
C_CUT = 41

# The recession in the generator: the macro path starts to rise after this calendar month.
RECESSION_ONSET = 44

# How a loan's record ends. `event` holds one of these.
OPEN, DEFAULT, PREPAID, MATURED = 0, 1, 2, 3
EVENT_NAMES = {OPEN: "still open at the data cut-off (censored)", DEFAULT: "defaulted",
               PREPAID: "prepaid in full", MATURED: "reached the end of its 36-month term"}


def _show(fig: "matplotlib.figure.Figure") -> None:
    """Display a figure in Jupyter, or close it cleanly in a headless script run."""
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("risk_table",),
    "exercise 2": ("vintage_curves",),
    "exercise 3": ("oot_masks",),
    "exercise 4": ("design_matrix",),
    "exercise 5": ("binomial_nll_grad",),
    "exercise 6": ("gradient_check",),
    "exercise 7": ("fit_newton",),
    "exercise 8": ("actual_vs_expected",),
    "exercise 9": ("term_structure",),
    "exercise 10": ("expected_credit_loss",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run
S: dict[str, Any] = {}         # results the demos hand to one another (tables, fits)


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (oot_masks)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# How long an exercise's check may run before it counts as stuck, and how long any other cell that runs your
# code (a demonstration, a table) may: a right answer needs a small fraction of either, measured on the worked
# solution (internal/retrofit_guard.py, 10x or more). tools/verify_all.py reads both numbers.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 10


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: int, memory_mib: float = 2048):
    """Turn a check that runs far too long into a failure, instead of a notebook that hangs.

    Uses the operating system's alarm, so it works where that exists (Linux, macOS, Colab) and is
    skipped, harmlessly, where it does not. An alarm that was already running (the grader's own
    time limit) keeps its deadline if it is due first and is put back, with the time it had
    left, when the check ends.
    """
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"still running after {seconds} s. Look for a loop whose stopping condition can never "
               "be met: a Newton loop with no iteration limit, or a counter that is never advanced")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def sigmoid(x: Any) -> Any:
    """Numerically safe logistic function 1 / (1 + exp(-x)). Given to you."""
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    pos = x >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-x[pos]))
    ex = np.exp(x[~pos])
    out[~pos] = ex / (1.0 + ex)
    return out[()]

## 1. The loan book, and the two clocks it runs on

Below is the generator. Read it: you are allowed to know exactly how this world works, which
is the one advantage a synthetic book gives you over real data.

Two clocks matter, and confusing them is the root of most of the mistakes in this lesson.

- **Age** (months on book) is the loan's own clock. A loan of age `a` has made `a` payments.
  Default risk has a shape in age: it is low at first, peaks in the second year, then falls.
- **Calendar month** is the economy's clock. A loan of age `a` from vintage `v` is
  living through calendar month `v + a`, and the macro variable at that month is what it
  experiences.

The hazard, the chance that a loan alive at the start of month `a` defaults during it, is

`logit(h) = alpha[a] + beta * stress[v + a] + gamma[v]`

where `alpha` is the age profile, `beta` the macro sensitivity, and `gamma[v]` a **hidden
vintage quality**: underwriting loosens part-way through the book, a little more each month.
The model you build will not be given `gamma`. Whether it notices is the back-test's job.
Each month a loan that has not defaulted may also **prepay**, which ends its record without a
default. A prepaid loan is not a good loan, and not a bad one: it is gone.

**Default** needs a definition before anything is counted. This lesson borrows the Basel
framework's reference one: a borrower is in default when the bank considers it unlikely to
pay in full without recourse to security, or when it is more than 90 days past due on a
material obligation (CRE36.68, as read on 14 October 2026; for retail exposures a
supervisor may substitute up to 180 days). A loan's first month in default is its default
event, and default is absorbing here: the record ends.

In [ ]:
class Book(NamedTuple):
    orig: np.ndarray        # (n_loans,) vintage: the month the loan was made
    exit_age: np.ndarray    # (n_loans,) age in months at which the record ends, 1..MAX_AGE
    event: np.ndarray       # (n_loans,) OPEN, DEFAULT, PREPAID or MATURED
    stress: np.ndarray      # (T_END + 1,) macro stress by calendar month
    gamma: np.ndarray       # (N_VINTAGES,) the hidden vintage quality: for checking only


BETA_TRUE = 0.30            # per percentage point of stress, on the logit scale
PREPAY_MONTHLY = 0.012      # chance a live, non-defaulting loan prepays in a month


def macro_path(seed: int = SEED) -> np.ndarray:
    """Unemployment-like rate (percent) for calendar months 0..T_END, then `stress`.

    A quiet, slowly wandering economy, then a recession that starts to bite just after
    RECESSION_ONSET, peaks a few months later and decays. The training window ends
    before it starts.
    """
    rng = np.random.default_rng(seed)
    t = np.arange(T_END + 1, dtype=float)
    noise = np.zeros(T_END + 1)
    shocks = rng.normal(0.0, 0.10, T_END + 1)
    for k in range(1, T_END + 1):
        noise[k] = 0.8 * noise[k - 1] + shocks[k]
    x = np.clip(t - RECESSION_ONSET, 0.0, None)
    recession = 7.5 * (1.0 - np.exp(-x / 3.0)) * np.exp(-x / 14.0)
    return 5.2 + 0.45 * np.sin(2.0 * np.pi * t / 30.0) + noise + recession


def simulate_book(seed: int = SEED) -> Book:
    """Deterministic synthetic loan book. Same seed, same book, on any machine.

    Every loan is followed month by month until its record ends: by default, by prepayment,
    by reaching age 36, or because the data stop at calendar month T_END with the loan still
    open. That last case is CENSORING. The loan did not stay good and did not go bad; we
    simply stopped looking.
    """
    rng = np.random.default_rng(seed + 1)
    ages = np.arange(1, MAX_AGE + 1)
    base = 0.0008 + 0.0065 * np.exp(-0.5 * ((ages - 13) / 6.0) ** 2)   # hump in age
    alpha = np.log(base / (1.0 - base))
    v = np.arange(N_VINTAGES)
    gamma = 0.02 * np.clip(v - 25, 0, None) + rng.normal(0.0, 0.04, N_VINTAGES)
    stress = macro_path(seed) - U_REF

    orig = np.repeat(v, LOANS_PER_VINTAGE)
    n = orig.size
    observable = np.minimum(MAX_AGE, T_END - orig)      # ages this loan can be watched for
    alive = np.ones(n, dtype=bool)
    exit_age = np.zeros(n, dtype=int)
    event = np.zeros(n, dtype=int)
    for age in range(1, MAX_AGE + 1):
        in_window = alive & (age <= observable)
        month = np.minimum(orig + age, T_END)           # beyond T_END nobody is watched
        hazard = sigmoid(alpha[age - 1] + BETA_TRUE * stress[month] + gamma[orig])
        defaults = in_window & (rng.random(n) < hazard)
        prepays = in_window & ~defaults & (rng.random(n) < PREPAY_MONTHLY)
        exit_age[defaults | prepays] = age
        event[defaults] = DEFAULT
        event[prepays] = PREPAID
        alive &= ~(defaults | prepays)
        last = alive & (age == observable)              # the record ends here, unresolved
        exit_age[last] = age
        event[last] = MATURED if age == MAX_AGE else OPEN
        alive &= ~last
    return Book(orig, exit_age, event, stress, gamma)


BOOK = simulate_book()
ORIG, EXIT_AGE, EVENT, STRESS = BOOK.orig, BOOK.exit_age, BOOK.event, BOOK.stress
print(f"{ORIG.size:,} loans in {N_VINTAGES} vintages, {LOANS_PER_VINTAGE:,} each; "
      f"data run to calendar month {T_END}")
for _code, _name in EVENT_NAMES.items():
    _k = int((EVENT == _code).sum())
    print(f"  {_name:<44} {_k:>7,}  ({100 * _k / ORIG.size:4.1f}%)")
print(f"\nthe last vintage is only {int(EXIT_AGE[ORIG == N_VINTAGES - 1].max())} months old at the "
      f"cut-off, against a term of {MAX_AGE}")

Here is the economy the book lived through, and where the training window stops. The model
you fit will be built at the dashed line, with no recession in its data.

In [ ]:
_fig, _ax = plt.subplots(figsize=(9, 2.6))
_ax.plot(np.arange(T_END + 1), STRESS + U_REF, lw=1.5, color="tab:red")
_ax.axvline(C_CUT, ls="--", color="k", lw=1)
_ax.text(C_CUT - 0.5, float((STRESS + U_REF).max()), "model built here", ha="right", va="top",
         fontsize=8)
_ax.axhline(U_REF, color="grey", lw=0.6)
_ax.set_xlabel("calendar month")
_ax.set_ylabel("unemployment-like rate, %")
_fig.tight_layout()
_show(_fig)
print(f"the rate spans {float((STRESS + U_REF)[:C_CUT + 1].min()):.1f}%-"
      f"{float((STRESS + U_REF)[:C_CUT + 1].max()):.1f}% up to the cut, then reaches "
      f"{float((STRESS + U_REF).max()):.1f}%")

## 2. Exercise 1 — `risk_table()`

Every later step starts from two tables with one row per vintage and one column per age:

- `at_risk[v, a-1]`: how many loans of vintage `v` were **observed for month `a`** while not
  yet defaulted, and so could have defaulted in it;
- `defaults[v, a-1]`: how many of them did.

The rule is the whole exercise. A loan whose record ends at age `e` was in the risk set for
months `1..e`, whatever the reason its record ended. If it defaulted, the default is counted
in month `e`. A prepaid or matured loan leaves the risk set after month `e`, having survived
it. And a **censored** loan, still open at the cut-off, was observed to survive every month
it was watched: it counts as at risk for those months and it is never a default. The classic
bug is to drop censored loans "because we do not know how they ended". They are the youngest
and healthiest part of the book, and dropping them inflates every later hazard.

<details><summary>💡 Hint 1 — what to think about</summary>

Take one loan that defaults at age 5 and one that is still open at age 5. In which months is
each one "at risk"? Are the answers different? Then ask what a table indexed by exit age
alone can tell you about a loan's earlier months.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Count how many loans of each vintage end their record at each age, then turn "ends at age
e" into "still observed at age a" by accumulating from the last age backwards. Count
defaults separately, only where the event is a default. Reject loans whose vintage, exit age
or event code is impossible, with `ValueError`, before counting anything.
</details>

In [ ]:
def risk_table(orig: np.ndarray, exit_age: np.ndarray, event: np.ndarray,
               n_vintages: int, max_age: int) -> tuple[np.ndarray, np.ndarray]:
    """Loans at risk, and defaults, by vintage and age. Both have shape (n_vintages, max_age).

    Column `a - 1` is age `a`. A loan whose record ends at `exit_age` is at risk in every
    month `1..exit_age`, whether it defaulted, prepaid, matured or was censored. A default is
    counted once, in month `exit_age`, and only when `event == DEFAULT`. Censored loans
    (`event == OPEN`) are at risk but are never defaults.

    Raises ValueError if the three arrays differ in length, a vintage is outside
    `0..n_vintages-1`, an exit age is outside `1..max_age`, or an event is not 0, 1, 2 or 3.

    Example (two vintages, terms of three months; the second loan matured, the third is open):
        >>> at, d = risk_table(np.array([0, 0, 1]), np.array([2, 3, 2]), np.array([1, 3, 0]),
        ...                    n_vintages=2, max_age=3)
        >>> at.tolist()
        [[2, 2, 1], [1, 1, 0]]
        >>> d.tolist()
        [[0, 1, 0], [0, 0, 0]]

    Returns:
        (at_risk, defaults): two integer arrays of shape (n_vintages, max_age).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_risk_table() -> None:
    at, d = risk_table(np.array([0, 0, 1]), np.array([2, 3, 2]), np.array([DEFAULT, MATURED, OPEN]),
                       n_vintages=2, max_age=3)
    assert np.asarray(at).shape == (2, 3) and np.asarray(d).shape == (2, 3), (
        f"expected two (n_vintages, max_age) = (2, 3) arrays, got {np.asarray(at).shape} and "
        f"{np.asarray(d).shape}")
    assert np.asarray(at).tolist() == [[2, 2, 1], [1, 1, 0]], (
        f"at_risk came back {np.asarray(at).tolist()}, expected [[2, 2, 1], [1, 1, 0]]. Vintage "
        "1's only loan is censored at age 2: it was observed for months 1 and 2, so it is at "
        "risk in both. A row of zeros there means censored loans were dropped; counting only "
        "loans whose exit age EQUALS the month loses every loan's earlier months")
    assert np.asarray(d).tolist() == [[0, 1, 0], [0, 0, 0]], (
        f"defaults came back {np.asarray(d).tolist()}, expected [[0, 1, 0], [0, 0, 0]]. The "
        "default is counted once, in its exit month, and the censored loan is not a default")
    at2, d2 = risk_table(np.array([0, 0]), np.array([1, 3]), np.array([PREPAID, DEFAULT]),
                         n_vintages=1, max_age=3)
    assert np.asarray(d2).tolist() == [[0, 0, 1]], (
        f"a prepaid loan is not a default: got defaults {np.asarray(d2).tolist()}")
    assert np.asarray(at2).tolist() == [[2, 1, 1]], (
        f"a prepaid loan is at risk in its exit month and gone after it: got "
        f"{np.asarray(at2).tolist()}, expected [[2, 1, 1]]")
    try:
        risk_table(np.array([0]), np.array([4]), np.array([0]), n_vintages=1, max_age=3)
    except ValueError:
        pass
    else:
        raise AssertionError("an exit age beyond max_age must raise ValueError, not be clipped")
    print("exercise 1 looks right — censored loans stay in the risk set and out of the defaults")

In [ ]:
_try("exercise 1", _check_risk_table)

In [ ]:
_FOR_TABLES = ("exercise 1",)
CAL_GRID_V, CAL_GRID_A = np.meshgrid(np.arange(N_VINTAGES), np.arange(1, MAX_AGE + 1),
                                     indexing="ij")
CAL = CAL_GRID_V + CAL_GRID_A          # calendar month of every (vintage, age) cell
OBSERVED = CAL <= T_END                # was that cell ever watched?


def _build_tables() -> None:
    at, d = risk_table(ORIG, EXIT_AGE, EVENT, N_VINTAGES, MAX_AGE)
    S["at"], S["d"] = np.asarray(at), np.asarray(d)
    at, d = S["at"], S["d"]
    print(f"risk table: {at.shape[0]} vintages x {at.shape[1]} ages = {at.size} cells, of which "
          f"{int(OBSERVED.sum())} were ever observed and {int((~OBSERVED).sum())} lie beyond "
          "the data cut-off")
    print(f"loan-months at risk: {int(at.sum()):,}  ·  defaults: {int(d.sum()):,}")
    for v in (0, N_VINTAGES - 1):
        row = "  ".join(f"age {a:>2}: {int(at[v, a - 1]):>5,}" for a in (1, 12, 24, 36))
        print(f"vintage {v:>2} at risk   {row}")
    print("\nthe zeros in the last vintage's row are not loans that all defaulted or prepaid.")
    print("they are months nobody has lived yet, and the next exercise must not confuse the two.")


_try("risk table", _build_tables, needs=_FOR_TABLES)

## 3. Exercise 2 — `vintage_curves()`

A **vintage curve** is the cumulative default rate of one cohort by months on book. Reading
a stack of them side by side is a standard credit-risk picture. A cohort that runs above its
neighbours was underwritten worse, or met a worse economy at a vulnerable age, and the curve
alone cannot say which: separating the two is what the hazard model below is for.

The estimator is the product limit. From your two tables, the hazard of a cell is
`defaults / at_risk`, and the curve is `1 - product(1 - hazard)` up to that age. Because
prepaid loans leave the risk set rather than counting as defaults, this is the default rate
**if the loans had all stayed**, the "net" convention a lifetime PD uses.

The point of the exercise is what the function does *not* return. Ages a cohort has not
reached by the data cut-off (cell `(v, a)` is observed only if `v + a <= t_end`) are
**not observed**, so they come back as `nan`: not the last value carried forward, and not
zero. Carrying the curve forward is the same mistake as counting censored loans as good.

<details><summary>💡 Hint 1 — what to think about</summary>

Suppose 100 loans start, 20 prepay in month 1 and 5 of the remaining 80 default in month 2.
What fraction of the original 100 defaulted by month 2, and what is the product-limit
answer? Which one would you defend, and what does dividing by the original count assume
about the prepaid loans?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Form the hazard cell by cell, using zero where nobody was at risk so nothing divides by
zero. Accumulate `1 - hazard` along the age axis and subtract from one. Then decide, from
the calendar month of each cell, which cells were ever watched, and blank exactly those that
were not.
</details>

In [ ]:
def vintage_curves(at_risk: np.ndarray, defaults: np.ndarray, t_end: int) -> np.ndarray:
    """Cumulative default rate by vintage and age, with unobserved cells set to NaN.

    `at_risk` and `defaults` have shape (n_vintages, max_age); row `v` is the vintage made in
    month `v`, column `a - 1` is age `a`. The hazard of a cell is `defaults / at_risk`, taken
    as 0.0 where `at_risk == 0`. The curve is `1 - cumprod(1 - hazard)` along the age axis.
    A cell is observed only if `v + a <= t_end`; every other cell is NaN.

    Example (vintage 1 is only two months old at t_end = 3):
        >>> at = np.array([[4, 3, 1], [4, 3, 0]])
        >>> d = np.array([[1, 1, 0], [0, 1, 0]])
        >>> np.round(vintage_curves(at, d, t_end=3), 3).tolist()
        [[0.25, 0.5, 0.5], [0.0, 0.333, nan]]

    Returns:
        A float array of shape (n_vintages, max_age): non-decreasing along each row where it
        is defined, with NaN wherever the cell was never observed.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_vintage_curves() -> None:
    at = np.array([[4, 3, 1], [4, 3, 0]])
    d = np.array([[1, 1, 0], [0, 1, 0]])
    got = np.asarray(vintage_curves(at, d, t_end=3), dtype=float)
    assert got.shape == (2, 3), f"expected shape (2, 3), got {got.shape}"
    assert np.isnan(got[1, 2]) and not np.isnan(got[1, 1]), (
        "vintage 1 is only two months old when the data stop at month 3, so its age-3 cell "
        f"must be NaN and its age-2 cell a number. Got {np.round(got[1], 3).tolist()} — a "
        "curve carried forward over months nobody watched treats them as clean")
    assert np.allclose(got[0], [0.25, 0.5, 0.5]), (
        f"vintage 0 came back {np.round(got[0], 4).tolist()}, expected [0.25, 0.5, 0.5]. The "
        "curve is 1 - product(1 - hazard), with each hazard measured against the loans STILL "
        "AT RISK: 1 default out of 3 in month 2 is 1/3, not 1/4 of the original loans")
    assert np.allclose(got[1, :2], [0.0, 1 / 3]), (
        f"vintage 1 came back {np.round(got[1, :2], 4).tolist()}, expected [0.0, 0.3333]")
    big = np.asarray(vintage_curves(np.array([[10, 5]]), np.array([[5, 3]]), t_end=9))
    assert np.allclose(big, [[0.5, 1 - 0.5 * 0.4]]), (
        f"hazards of 0.5 then 0.6 must give [0.5, 0.8]; got {np.round(big, 4).tolist()}. "
        "Adding the hazards instead of multiplying the survival factors overshoots")
    print("exercise 2 looks right — a vintage curve stops where the data stop")

In [ ]:
_try("exercise 2", _check_vintage_curves)

Now the classic mistake, next to the right answer. The "naive" number below is what a query
written in a hurry returns: for every loan, a 1 if it defaulted within 24 months and a 0
otherwise, averaged over the vintage. It treats a loan that is still open, or that has not
yet reached month 24, as a good loan.

In [ ]:
_FOR_CURVES = _FOR_TABLES + ("exercise 2",)


def _show_naive_vs_curve() -> None:
    at, d = S["at"], S["d"]
    S["curves"] = vintage_curves(at, d, T_END)
    horizon = 24
    naive = d[:, :horizon].sum(axis=1) / at[:, 0]      # defaults within 24 months / originated
    print(f"cumulative default rate by month {horizon}, vintage by vintage")
    print(f"{'vintage':>8} {'watched for':>12} {'naive':>8} {'product limit':>14} {'naive / right':>14}")
    for v in (0, 12, 24, 30, 36, 40, 42, 44, 47):
        watched = int(min(MAX_AGE, T_END - v))
        right = S["curves"][v, horizon - 1]
        if np.isnan(right):
            print(f"{v:>8} {watched:>9} mo {naive[v]:>8.3f} {'not observed':>14} {'':>14}")
        else:
            print(f"{v:>8} {watched:>9} mo {naive[v]:>8.3f} {right:>14.3f} {naive[v] / right:>14.2f}")
    unobserved = int(np.isnan(S["curves"][:, horizon - 1]).sum())
    print(f"\n{unobserved} of {N_VINTAGES} vintages have not been watched for {horizon} months. The naive "
          "column gives every one of them a number,")
    print("but it counts defaults over fewer than 24 months, so it can only understate the "
          "24-month rate, by an amount nobody can know yet.")
    seen = ~np.isnan(S["curves"][:, horizon - 1])
    below = int((naive[seen] < S["curves"][seen, horizon - 1]).sum())
    print(f"on the {int(seen.sum())} vintages that were watched that long the naive rate is below the "
          f"right one for {below} of them:")
    print("a prepaid loan is scored as a survivor forever, so every rate is diluted.")


_try("naive versus curve", _show_naive_vs_curve, needs=_FOR_CURVES)

In [ ]:
def _plot_vintage_fan() -> None:
    curves = S["curves"]
    fig, ax = plt.subplots(figsize=(9, 3.6))
    cmap = plt.get_cmap("viridis")
    for v in range(N_VINTAGES):
        ax.plot(np.arange(1, MAX_AGE + 1), 100 * curves[v], lw=1.0, color=cmap(v / (N_VINTAGES - 1)))
    ax.set_xlabel("months on book")
    ax.set_ylabel("cumulative default rate, %")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, N_VINTAGES - 1))
    fig.colorbar(sm, ax=ax, label="vintage (origination month)")
    fig.tight_layout()
    _show(fig)
    v_lo, v_hi = 12, 40
    print(f"at month 12 the curve of vintage {v_lo} sits at {100 * curves[v_lo, 11]:.1f}%; "
          f"vintage {v_hi}'s sits at {100 * curves[v_hi, 11]:.1f}%.")
    print("each line stops where its vintage's data stop; nothing is extrapolated to fill the "
          "gap.")


_try("vintage fan", _plot_vintage_fan, needs=_FOR_CURVES)

## 4. Exercise 3 — `oot_masks()`

A model is judged on data it has not seen, and in lending the phrase has a precise meaning:
not seen **and later**. The test vintages are made after the training vintages, so the model
meets borrowers, underwriting and an economy it could not have been fitted to. A random
split of loans does the opposite: every test loan has training siblings from the same
vintage and the same calendar months, so the model has been shown its own future.

The protocol here has two cuts, and both are the exercise:

- **by vintage:** training cells belong to vintages `< v_oot`; test cells to vintages
  `>= v_oot`, and nothing else;
- **by calendar:** a model built at calendar month `c_cut` cannot have seen any cell from a
  later month, even from an old vintage, so training cells also need `calendar <= c_cut`.

<details><summary>💡 Hint 1 — what to think about</summary>

A loan from vintage 3 that is still alive in calendar month 90: is it a training loan or a
test loan for a model built at month 60 that is tested on vintages from 50? Which of its
months could the model have seen? Then check which side each boundary value falls on.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Each mask is a comparison on the vintage array, and the training mask also compares the
calendar array. The test mask must not mention the calendar at all: test vintages are
scored on everything that was ever observed of them. Return two boolean arrays with the
shape of the inputs.
</details>

In [ ]:
def oot_masks(vintage: np.ndarray, calendar: np.ndarray, v_oot: int,
              c_cut: int) -> tuple[np.ndarray, np.ndarray]:
    """Train and test masks for an out-of-time split by vintage, with a calendar cut.

    `vintage` and `calendar` give, for every cell, the vintage it belongs to and the calendar
    month it was lived in (any matching shape). A cell is TRAIN if `vintage < v_oot` and
    `calendar <= c_cut`. A cell is TEST if `vintage >= v_oot`, whatever its calendar month.
    No cell is both, and a training vintage's cells after `c_cut` are neither: they are
    the future of the model.

    Example:
        >>> tr, te = oot_masks(np.array([29, 30, 30]), np.array([41, 41, 42]), 30, 41)
        >>> tr.tolist(), te.tolist()
        ([True, False, False], [False, True, True])

    Returns:
        (train, test): boolean arrays with the shape of `vintage`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_oot_masks() -> None:
    v = np.array([29, 30, 30, 5, 5, 47])
    c = np.array([41, 41, 42, 41, 42, 60])
    tr, te = oot_masks(v, c, 30, 41)
    tr, te = np.asarray(tr), np.asarray(te)
    assert tr.dtype == bool and te.dtype == bool and tr.shape == v.shape, (
        "both masks must be boolean arrays with the shape of the inputs")
    assert tr.tolist() == [True, False, False, True, False, False], (
        f"train came back {tr.tolist()}, expected [T, F, F, T, F, F]. Vintage 30 is the first "
        "TEST vintage, and month 41 is the last month a model built at 41 has seen: cell "
        "(vintage 5, month 42) is the future even though vintage 5 is a training vintage")
    assert te.tolist() == [False, True, True, False, False, True], (
        f"test came back {te.tolist()}, expected [F, T, T, F, F, T]. Test means vintage >= "
        "v_oot, and it is scored on every month of those vintages, early or late")
    assert not (tr & te).any(), "no cell may be in both train and test"
    print("exercise 3 looks right — the model is built at a date and tested on later vintages")

In [ ]:
_try("exercise 3", _check_oot_masks)

In [ ]:
_FOR_SPLIT = _FOR_TABLES + ("exercise 3",)


def _show_split() -> None:
    at, d = S["at"], S["d"]
    usable = OBSERVED & (at > 0)
    tr, te = oot_masks(CAL_GRID_V, CAL, V_OOT, C_CUT)
    S["train"], S["test"] = np.asarray(tr) & usable, np.asarray(te) & usable
    for name, m in (("train", S["train"]), ("test", S["test"])):
        vs = CAL_GRID_V[m]
        print(f"{name:>5}: {int(m.sum()):>5} cells, vintages {int(vs.min())}..{int(vs.max())}, "
              f"calendar months {int(CAL[m].min())}..{int(CAL[m].max())}, "
              f"{int(d[m].sum()):>5,} defaults")
    print(f"the recession starts after month {RECESSION_ONSET}; the model's data end at month "
          f"{C_CUT}.")
    lives = int((CAL[S["test"]] > RECESSION_ONSET).sum())
    print(f"{lives} of the test cells lie inside it, and {int((CAL[S['train']] > RECESSION_ONSET).sum())} "
          "of the training cells do.")


_try("split", _show_split, needs=_FOR_SPLIT)

## 5. Exercise 4 — `design_matrix()`

Now the model. A **discrete-time hazard model** is a logistic regression on person-months:
each row of the design matrix is one (vintage, age) cell, the response is "defaults out of
loans at risk", and the covariates say what the cell is like.

Two of them, and both matter:

- **an intercept per month on book.** Default risk has a shape in age, low then a hump then
  a decline. One global intercept plus an age slope cannot draw a hump; a separate intercept
  for every age can draw any shape. With one column per age there is **no separate constant
  column**: it would equal the sum of the age columns, and the fit would be singular.
- **the macro stress** of the calendar month the cell was lived in, one number per cell.

<details><summary>💡 Hint 1 — what to think about</summary>

For a cell of age 3 in a model with ages up to 36, which columns should be 1? How many
columns are there in total, and what would happen to the matrix's rank if you also added a
column of ones?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Start from an all-zero array with one row per cell and `max_age + 1` columns. Put a single
one per row in the column of that row's age (ages count from 1, columns from 0), and copy
the stress into the last column. Reject an age outside `1..max_age` with `ValueError`.
</details>

In [ ]:
def design_matrix(age: np.ndarray, stress: np.ndarray, max_age: int) -> np.ndarray:
    """Design matrix of a hazard model with one intercept per age and one macro slope.

    Row `i` describes one cell: `age[i]` (1-based, in `1..max_age`) and `stress[i]`. Columns
    `0..max_age-1` are one-hot age indicators (column `a - 1` is 1 when the age is `a`), and
    the last column is the stress. There is no separate constant column.

    Raises ValueError if any age is outside `1..max_age` or the lengths differ.

    Example:
        >>> design_matrix(np.array([1, 3]), np.array([0.5, -1.0]), max_age=3).tolist()
        [[1.0, 0.0, 0.0, 0.5], [0.0, 0.0, 1.0, -1.0]]

    Returns:
        A float array of shape (len(age), max_age + 1).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_design_matrix() -> None:
    x = np.asarray(design_matrix(np.array([1, 3, 2]), np.array([0.5, -1.0, 2.0]), max_age=3))
    assert x.shape == (3, 4), (
        f"expected (3, 4) = one column per age plus one for the stress, got {x.shape}. An "
        "extra column of ones makes the age columns sum to it: the matrix is then singular")
    assert x.tolist() == [[1, 0, 0, 0.5], [0, 0, 1, -1.0], [0, 1, 0, 2.0]], (
        f"got {x.tolist()}. Age 1 is column 0, age 3 is column 2, and the stress is the last "
        "column, unscaled")
    assert (x[:, :3].sum(axis=1) == 1).all(), "every row must have exactly one age indicator on"
    try:
        design_matrix(np.array([4]), np.array([0.0]), max_age=3)
    except ValueError:
        pass
    else:
        raise AssertionError("an age beyond max_age must raise ValueError, not index off the end")
    print("exercise 4 looks right — one intercept per month on book, plus the macro slope")

In [ ]:
_try("exercise 4", _check_design_matrix)

## 6. Exercise 5 — `binomial_nll_grad()`

Fitting the model means minimising a loss. Each cell has `n` loans at risk, `d` of which
defaulted, and the model says each defaults with probability `p = sigmoid(eta)` where
`eta = X @ beta`. The **negative log-likelihood** of the whole table, dropping a constant
that does not depend on `beta`, is

`nll = sum( n * log(1 + exp(eta)) - d * eta )`

and its gradient is `X.T @ (n * p - d)`: each cell pulls `beta` by the gap between the
defaults the model expects and the defaults it saw. Two things go wrong in practice. Written
naively, `log(1 + exp(eta))` overflows to infinity once `eta` reaches a few hundred, and a
fit that wanders there gets an infinite loss and no direction. And people average instead of
summing, which quietly changes what the step size and the convergence tolerance mean.

<details><summary>💡 Hint 1 — what to think about</summary>

What does `log(1 + exp(eta))` do for `eta = 800`? NumPy has a function that computes
`log(exp(a) + exp(b))` without ever forming either exponential; which values of `a` and `b`
give the term you need?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Form `eta`, evaluate the loss with the overflow-safe log-sum-exp, and sum over cells without
dividing by anything. The gradient is one matrix product between the transposed design
matrix and the vector of expected-minus-observed defaults. Reject `d > n` or a negative
count with `ValueError`: no table can hold more defaults than loans at risk.
</details>

In [ ]:
def binomial_nll_grad(beta: np.ndarray, X: np.ndarray, n: np.ndarray,
                      d: np.ndarray) -> tuple[float, np.ndarray]:
    """Negative log-likelihood of a grouped logistic model, and its gradient.

    With `eta = X @ beta` and `p = sigmoid(eta)`:
        nll  = sum(n * log(1 + exp(eta)) - d * eta)        (a SUM, not a mean)
        grad = X.T @ (n * p - d)
    `n` (loans at risk) and `d` (defaults) are per-row counts with `0 <= d <= n`; otherwise
    raise ValueError. The loss must stay finite when `|eta|` is in the hundreds.

    Example (two cells, `beta = 0` so every p is one half):
        >>> nll, g = binomial_nll_grad(np.zeros(2), np.eye(2), np.array([10.0, 10.0]),
        ...                            np.array([3.0, 5.0]))
        >>> round(nll, 4), g.tolist()
        (13.8629, [2.0, 0.0])

    Returns:
        (nll, grad): a Python float and an array with the shape of `beta`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_binomial_nll_grad() -> None:
    nll, g = binomial_nll_grad(np.zeros(2), np.eye(2), np.array([10.0, 10.0]), np.array([3.0, 5.0]))
    assert type(nll) is float, (
        f"the loss must be a plain Python float, got {type(nll).__name__}")
    assert abs(nll - 20 * np.log(2)) < 1e-9, (
        f"at beta = 0 every p is 1/2 and the loss is sum(n) * log 2 = {20 * np.log(2):.4f}; "
        f"got {nll:.4f}. Averaging over cells, or using the deviance, changes this")
    assert np.allclose(g, [2.0, 0.0]), (
        f"gradient came back {np.asarray(g).tolist()}, expected [2, 0] = X.T @ (n * 0.5 - d). "
        "The sign is expected minus observed, and n multiplies the probability")
    big, _ = binomial_nll_grad(np.array([900.0]), np.ones((1, 1)), np.array([1.0]), np.array([0.0]))
    assert np.isfinite(big) and abs(big - 900.0) < 1e-6, (
        f"eta = 900 must give a finite loss of about 900, got {big}. log(1 + exp(eta)) "
        "overflows; use np.logaddexp(0, eta)")
    try:
        binomial_nll_grad(np.zeros(1), np.ones((1, 1)), np.array([1.0]), np.array([2.0]))
    except ValueError:
        pass
    else:
        raise AssertionError("more defaults than loans at risk must raise ValueError")
    print("exercise 5 looks right — a stable loss and the gradient that goes with it")

In [ ]:
_try("exercise 5", _check_binomial_nll_grad)

## 7. Exercise 6 — `gradient_check()`

A gradient that is wrong by a sign, a factor or a missing weight still lets a fit run and
still produces coefficients. They are just not the maximum-likelihood ones, and nothing
downstream will tell you. So check the gradient against the thing it claims to be: the slope
of the loss. The **central difference**

`(f(beta + eps * e_i) - f(beta - eps * e_i)) / (2 * eps)`

estimates the i-th partial derivative with an error that shrinks with `eps` squared, where a
one-sided difference shrinks only with `eps`. Report the worst coordinate as a *relative*
error, `|analytic - numeric| / max(1, |analytic|, |numeric|)`, so that big gradients are
judged by proportion and near-zero ones by the absolute gap.

<details><summary>💡 Hint 1 — what to think about</summary>

You must perturb one coordinate at a time. What should happen to `beta` itself while you do
it: does the caller expect to get the same array back, unchanged?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Call the function once at `beta` for the analytic gradient. Then for each coordinate, take
the loss at a point `eps` above and a point `eps` below along that axis and divide the
difference by `2 * eps`. Compare with the analytic value coordinate by coordinate and return
the largest relative error as a float.
</details>

In [ ]:
def gradient_check(fun: Callable[[np.ndarray], tuple[float, np.ndarray]], beta: np.ndarray,
                   eps: float = 1e-6) -> float:
    """Worst relative disagreement between an analytic gradient and central differences.

    `fun(beta)` returns `(loss, gradient)`. For each coordinate `i`, the numeric derivative is
    `(loss(beta + eps * e_i) - loss(beta - eps * e_i)) / (2 * eps)`. The result is
        max_i  |analytic_i - numeric_i| / max(1, |analytic_i|, |numeric_i|).
    `beta` must be left exactly as it was given.

    Example (loss = half the squared norm, whose gradient is beta itself):
        >>> f = lambda b: (0.5 * float(b @ b), b.copy())
        >>> gradient_check(f, np.array([1.0, -2.0])) < 1e-8
        True

    Returns:
        A Python float, the worst relative error.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_gradient_check() -> None:
    quad = lambda b: (0.5 * float(b @ b), b.copy())          # noqa: E731
    beta = np.array([1.0, -2.0, 0.5])
    keep = beta.copy()
    good = gradient_check(quad, beta)
    assert type(good) is float and good < 1e-7, (
        f"the check on a correct gradient should be tiny, got {good!r}. Central differences "
        "divide by 2 * eps, not eps")
    assert np.array_equal(beta, keep), "gradient_check must not modify the beta it is given"
    wrong = lambda b: (0.5 * float(b @ b), 1.05 * b)         # noqa: E731
    bad = gradient_check(wrong, beta)
    assert bad > 1e-3, (
        f"a gradient that is 5% too big must be flagged, but the check returned {bad:.2e}")
    print("exercise 6 looks right — it passes a true gradient and flags a wrong one")

In [ ]:
_try("exercise 6", _check_gradient_check)

In [ ]:
def _show_gradient_check() -> None:
    at, d = S["at"], S["d"]
    m = S["train"]
    X = design_matrix(CAL_GRID_A[m], STRESS[CAL[m]], MAX_AGE)
    n, y = at[m].astype(float), d[m].astype(float)
    rng = np.random.default_rng(SEED)
    beta = rng.normal(0.0, 0.3, X.shape[1])
    ok = gradient_check(lambda b: binomial_nll_grad(b, X, n, y), beta)
    print(f"your gradient against central differences, on {int(m.sum())} real cells at a random "
          f"beta: error below 1e-6 is {ok < 1e-6}")
    # A gradient with the classic slip: it forgot that each cell has n loans at risk.
    no_weight = lambda b: (binomial_nll_grad(b, X, n, y)[0],                # noqa: E731
                           X.T @ (sigmoid(X @ b) - y))
    slip = gradient_check(no_weight, beta)
    print(f"the same check on a gradient that forgot the n weights: error below 1e-6 is "
          f"{slip < 1e-6}")
    print("a fit driven by the second one would run, converge to something, and be wrong.")


_try("gradient check demo", _show_gradient_check,
     needs=_FOR_SPLIT + ("exercise 4", "exercise 5", "exercise 6"))

## 8. Exercise 7 — `fit_newton()`

With a correct loss and gradient, fit the coefficients by **Newton's method**. At the current
`beta`, the curvature of the loss is `H = X.T @ diag(n * p * (1 - p)) @ X`; the Newton step
is the solution of `H @ step = gradient`, and `beta` moves by `-step`. Near the optimum each
step roughly doubles the number of correct digits, which is why a well-behaved logistic fit
finishes in a handful of iterations where plain gradient descent needs far more.

Two details keep it honest. Stop when the **Newton decrement** `gradient @ step` falls below
a tolerance: it is the loss you still expect to gain, so it does not depend on how the
problem is scaled. And a tiny ridge penalty, `0.5 * ridge * beta @ beta`, keeps `H`
invertible when some age has no defaults at all, where the unpenalised optimum sits at minus
infinity. Under the penalty the gradient is `gradient + ridge * beta` and `H` gains
`ridge * I`.

<details><summary>💡 Hint 1 — what to think about</summary>

Where should `beta` start? What does the fit look like for a column with no defaults: does
the coefficient converge, and how many Newton steps does that take? Look at your own step
count before you decide the loop limit is generous enough.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Each pass: get the loss and gradient from your exercise 5, add the ridge terms, build the
weighted curvature, solve for the step, and stop if the decrement is below `tol`.
Otherwise take the step; if the penalised loss got worse, halve the step until it does not.
Return the coefficients and the number of steps actually taken.
</details>

In [ ]:
def fit_newton(X: np.ndarray, n: np.ndarray, d: np.ndarray, ridge: float = 1e-8,
               max_iter: int = 50, tol: float = 1e-10) -> tuple[np.ndarray, int]:
    """Maximum-likelihood coefficients of a grouped logistic model, by Newton's method.

    Minimises `binomial_nll_grad(beta, X, n, d)[0] + 0.5 * ridge * (beta @ beta)` from
    `beta = 0`. Each step solves `H @ step = g` with `g = grad + ridge * beta` and
    `H = X.T @ diag(n * p * (1 - p)) @ X + ridge * I`. Stop when `g @ step < tol`. If a full
    step raises the penalised loss, halve it until it does not. Do not modify the inputs.

    Example (an intercept-only model: 6 defaults in 20 loans gives logit(0.3)):
        >>> beta, steps = fit_newton(np.ones((2, 1)), np.array([10.0, 10.0]), np.array([2.0, 4.0]))
        >>> round(float(beta[0]), 4), steps <= 10
        (-0.8473, True)

    Returns:
        (beta, n_steps): the coefficient array and the number of Newton steps taken.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_fit_newton() -> None:
    beta, steps = fit_newton(np.ones((2, 1)), np.array([10.0, 10.0]), np.array([2.0, 4.0]))
    assert np.asarray(beta).shape == (1,), f"beta should have shape (1,), got {np.asarray(beta).shape}"
    assert abs(float(beta[0]) - np.log(0.3 / 0.7)) < 1e-6, (
        f"6 defaults in 20 loans is p = 0.3, so beta = logit(0.3) = {np.log(0.3 / 0.7):.4f}; "
        f"got {float(beta[0]):.4f}")
    assert isinstance(steps, (int, np.integer)) and 1 <= steps <= 10, (
        f"Newton's method finishes this in a handful of steps; steps = {steps!r}. A count of "
        "0 means it stopped before moving; a large one means it is doing gradient descent")
    X = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0], [1.0, 3.0]])
    n = np.array([200.0, 200.0, 200.0, 200.0])
    p_true = sigmoid(-2.0 + 0.5 * X[:, 1])
    d = np.round(n * p_true)
    b, _ = fit_newton(X, n, d)
    _, g = binomial_nll_grad(b, X, n, d)
    assert np.abs(g).max() < 1e-4, (
        f"at your solution the gradient should vanish, but its largest entry is "
        f"{np.abs(g).max():.2e}: it has not converged")
    print("exercise 7 looks right — Newton's method, stopped on the decrement")

In [ ]:
_try("exercise 7", _check_fit_newton)

Fit the model on the training cells only (vintages before the cut, months up to the cut), and
look at what came out. Note what is *in* the fit and what is not: the macro slope has been
estimated from a quiet economy, and no vintage effect exists at all.

In [ ]:
_FOR_MODEL = _FOR_SPLIT + ("exercise 4", "exercise 5", "exercise 7")


def fit_on(at: np.ndarray, d: np.ndarray, mask: np.ndarray,
           extra: "np.ndarray | None" = None) -> tuple[np.ndarray, int]:
    """Fit the hazard model on the masked cells of a risk table. Given to you.

    `extra`, if supplied, is one more covariate per cell (shape of the grid), added as a last
    column: the demos use it to show what a leaky feature does. Returns `(coef, steps)`.
    """
    X = design_matrix(CAL_GRID_A[mask], STRESS[np.minimum(CAL[mask], T_END)], MAX_AGE)
    if extra is not None:
        X = np.hstack([X, extra[mask][:, None]])
    return fit_newton(X, at[mask].astype(float), d[mask].astype(float))


def _show_fit() -> None:
    at, d = S["at"], S["d"]
    m = S["train"]
    coef, steps = fit_on(at, d, m)
    S["coef_oot"] = coef
    X = design_matrix(CAL_GRID_A[m], STRESS[CAL[m]], MAX_AGE)
    _, g = binomial_nll_grad(coef, X, at[m].astype(float), d[m].astype(float))
    print(f"Newton steps taken: {steps}   ·   gradient at the solution below 1e-4: "
          f"{bool(np.abs(g).max() < 1e-4)}")
    print(f"macro slope, fitted on the quiet training window: {coef[-1]:.3f}   "
          f"(planted: {BETA_TRUE:.3f})")
    fitted_hump = sigmoid(coef[:MAX_AGE])
    by_age = lambda a: np.bincount(CAL_GRID_A[m], weights=a[m], minlength=MAX_AGE + 1)[1:]   # noqa: E731
    empirical = by_age(d) / np.maximum(by_age(at), 1)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(np.arange(1, MAX_AGE + 1), 100 * empirical, "o", ms=3, label="pooled empirical hazard")
    ax.plot(np.arange(1, MAX_AGE + 1), 100 * fitted_hump, lw=1.5,
            label="fitted, at an ordinary month (stress 0)")
    ax.set_xlabel("months on book")
    ax.set_ylabel("monthly default hazard, %")
    ax.legend(fontsize=8)
    fig.tight_layout()
    _show(fig)
    peak = int(np.argmax(fitted_hump)) + 1
    print(f"the fitted hazard peaks at month {peak}: one intercept per age drew the hump. "
          "A single intercept and an age slope could not.")


_try("fit demo", _show_fit, needs=_FOR_MODEL)

## 9. Exercise 8 — `actual_vs_expected()`

Supervisory guidance on model risk describes **outcomes analysis** as comparing "model
outputs to corresponding real-world outcomes", and names back-testing as one form of it (the
2026 interagency guidance, as read on 29 September 2026; source in `claims.yaml`). For a PD
model the plainest outcomes analysis is a ratio: **actual defaults over expected defaults**,
where the expected count of a cell is the loans at risk times the hazard the model gave it.
A ratio of 1.0 is calibrated. Above 1.0 the model was too optimistic; below, too cautious.

You compute it for whatever grouping you ask about: by vintage to see who was
mispredicted, by calendar window to see when. Two rules make it trustworthy:

- only cells in `mask` count, on both sides of the ratio. Cells nobody observed carry no
  defaults and no loans at risk, and the model may not even have a prediction for them (a
  `nan` there must not leak into a sum);
- a group with nothing expected has **no ratio**: report `nan`, never `inf`, never `0`.

<details><summary>💡 Hint 1 — what to think about</summary>

If you multiply `at_risk * hazard` over the whole grid and one unobserved cell holds a
`nan`, what happens to the sum of its group? Where must the mask be applied to prevent it?
And what should a group with no expected defaults report?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Select the masked cells first, from all four arrays, then total the defaults and the
expected defaults per group (a weighted bincount does it in one pass, with `minlength` so
empty groups still get a slot). Divide only where the expected total is positive.
</details>

In [ ]:
def actual_vs_expected(defaults: np.ndarray, at_risk: np.ndarray, hazard: np.ndarray,
                       group: np.ndarray, n_groups: int,
                       mask: "np.ndarray | None" = None) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Actual and expected defaults per group, and their ratio.

    All arrays have the same shape (any). A cell contributes `defaults` to its group's actual
    and `at_risk * hazard` to its group's expected, but only where `mask` is True (default:
    every cell). Cells outside the mask may hold NaN in `hazard`; they must not matter. The
    ratio is `actual / expected`, and NaN for a group whose expected total is 0. Groups are
    integers `0..n_groups-1`, and every group gets an entry.

    Example (three cells; group 2 has nothing in it):
        >>> a, e, r = actual_vs_expected(np.array([1, 1, 4]), np.array([10, 10, 10]),
        ...                              np.array([0.1, 0.2, 0.3]), np.array([0, 0, 1]), 3)
        >>> a.tolist(), e.tolist(), np.round(r, 4).tolist()
        ([2.0, 4.0, 0.0], [3.0, 3.0, 0.0], [0.6667, 1.3333, nan])

    Returns:
        (actual, expected, ratio): three float arrays of length `n_groups`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_actual_vs_expected() -> None:
    a, e, r = actual_vs_expected(np.array([1, 1, 4]), np.array([10, 10, 10]),
                                 np.array([0.1, 0.2, 0.3]), np.array([0, 0, 1]), 3)
    a, e, r = np.asarray(a), np.asarray(e), np.asarray(r)
    assert a.shape == e.shape == r.shape == (3,), (
        f"each result must have one entry per group, empty ones included; shapes {a.shape}, "
        f"{e.shape}, {r.shape}")
    assert np.allclose(a, [2, 4, 0]) and np.allclose(e, [3, 3, 0]), (
        f"actual {a.tolist()}, expected {e.tolist()}; want [2, 4, 0] and [3, 3, 0]. Expected "
        "defaults are at_risk * hazard summed within the group, not the hazards alone")
    assert np.allclose(r[:2], [2 / 3, 4 / 3]) and np.isnan(r[2]), (
        f"ratio came back {np.round(r, 4).tolist()}; an empty group has NO ratio (nan), not 0 "
        "and not inf")
    mask = np.array([True, True, False])
    a2, e2, r2 = actual_vs_expected(np.array([1, 1, 4]), np.array([10, 10, 0]),
                                    np.array([0.1, 0.2, np.nan]), np.array([0, 0, 1]), 2, mask)
    assert np.isfinite(e2).all() and np.allclose(e2, [3, 0]), (
        f"a NaN hazard in a masked-out cell must not reach the sums; expected {np.asarray(e2).tolist()}")
    assert np.isnan(np.asarray(r2)[1]) and np.asarray(a2)[1] == 0, (
        "the masked cell's 4 defaults belong to no group: group 1 has nothing left")
    print("exercise 8 looks right — actual over expected, group by group, on observed cells only")

In [ ]:
_try("exercise 8", _check_actual_vs_expected)

## 10. Back-testing through the recession

The model was built at the cut month from the vintages before the split. Below, its hazard is
applied to every test vintage over the months that vintage was actually observed, and compared with what
happened, vintage by vintage. The macro path fed in is the **realised** one, so this is a
conditional back-test: it asks "given the economy that happened, was the model right?".
That is a legitimate question, and section 12 is about how it gets mistaken for a different
one. The last column is the hidden vintage quality the generator planted, which the model
was never given.

In [ ]:
_FOR_BACKTEST = _FOR_MODEL + ("exercise 8",)


def hazard_grid(coef: np.ndarray, stress_by_month: np.ndarray = STRESS,
                extra: "np.ndarray | None" = None) -> np.ndarray:
    """Model hazard for every (vintage, age) cell, with the macro path you supply. Given.

    `stress_by_month` is indexed by calendar month; months beyond its end reuse its last value
    (those cells are unobserved and never scored). `extra` is an optional extra covariate;
    when it is used it is the LAST coefficient and the macro slope is then `coef[-2]`.
    """
    coef = np.asarray(coef)
    st = np.asarray(stress_by_month)[np.minimum(CAL, len(stress_by_month) - 1)]
    eta = coef[CAL_GRID_A - 1] + coef[MAX_AGE] * st
    if extra is not None:
        eta = eta + coef[MAX_AGE + 1] * extra
    return sigmoid(eta)


def backtest(coef: np.ndarray, at: np.ndarray, d: np.ndarray, mask: np.ndarray,
             stress_by_month: np.ndarray = STRESS, extra: "np.ndarray | None" = None):
    """(actual, expected, ratio) by vintage, plus the portfolio ratio and the spread. Given.

    The spread is the mean absolute log of the by-vintage ratio: 0.0 would be a model that is
    right about every vintage, and 0.22 means a typical vintage is off by about a quarter.
    """
    hz = hazard_grid(coef, stress_by_month, extra)
    a, e, r = actual_vs_expected(d, at, hz, CAL_GRID_V, N_VINTAGES, mask)
    ok = np.isfinite(r) & (a > 0)
    return a, e, r, float(a.sum() / e.sum()), float(np.mean(np.abs(np.log(r[ok]))))


def _show_backtest() -> None:
    at, d, te = S["at"], S["d"], S["test"]
    a, e, r, port, spread = backtest(S["coef_oot"], at, d, te)
    S["bt_honest"] = (port, spread)
    print(f"{'vintage':>8} {'actual':>8} {'expected':>9} {'A / E':>7} {'hidden quality':>15}")
    for v in range(V_OOT, N_VINTAGES):
        print(f"{v:>8} {a[v]:>8.0f} {e[v]:>9.1f} {r[v]:>7.2f} {BOOK.gamma[v]:>+15.3f}")
    print(f"\nportfolio A / E over the test vintages: {port:.2f}   ·   spread: {spread:.2f}")
    fig, ax = plt.subplots(figsize=(8, 2.8))
    ax.bar(np.arange(V_OOT, N_VINTAGES), r[V_OOT:], color="tab:blue")
    ax.axhline(1.0, color="k", lw=0.8)
    ax.set_xlabel("test vintage")
    ax.set_ylabel("actual / expected defaults")
    fig.tight_layout()
    _show(fig)
    print("A / E above 1.0 means the model under-predicted. Compare the last two columns: the "
          "generator's underwriting loosened,")
    print("the model has no term for it, and section 15 measures how closely the two move "
          "together.")


_try("back-test by vintage", _show_backtest, needs=_FOR_BACKTEST)

## 11. Point in time or through the cycle?

The same fitted model can give two different PDs, and they answer different questions.

- A **point-in-time (PIT)** PD uses the macro conditions of the month in question. It moves
  with the cycle, and it is what an expected-loss provision needs: the US current expected
  credit loss standard asks for reasonable and supportable forecasts on top of past events
  and current conditions (as read on 14 October 2026; sources in `claims.yaml`).
- A **through-the-cycle (TTC)** PD holds the macro at a long-run average. It moves with the
  cycle only through the age mix of the loans, and it is the flavour the Basel framework's
  internal-ratings approach asks for: a long-run average of one-year default rates
  (CRE36.63), and for retail exposures like these, data that include "a representative mix
  of good and bad years of the economic cycle" (CRE36.82; both as read on 14 October 2026).

Here they are, measured. Both are your fitted model, given a different stress input. The
"long-run average" available in this book is the mean stress in the training window, which is
a quiet stretch with no recession in it. Sit with that: a TTC PD calibrated on a window
without a downturn is not through any cycle.

In [ ]:
def _show_pit_vs_ttc() -> None:
    at, d, te = S["at"], S["d"], S["test"]
    coef = S["coef_oot"]
    train_mean = float(STRESS[CAL[S["train"]]].mean())
    hz_pit = hazard_grid(coef)
    hz_ttc = hazard_grid(coef, np.full(T_END + 1, train_mean))
    peak_end = RECESSION_ONSET + 12
    regime = np.where(CAL <= RECESSION_ONSET, 0, np.where(CAL <= peak_end, 1, 2))
    names = [f"quiet (to month {RECESSION_ONSET})", f"recession ({RECESSION_ONSET + 1}-{peak_end})",
             f"recovery ({peak_end + 1}+)"]
    rows = {}
    for label, hz in (("PIT", hz_pit), ("TTC", hz_ttc)):
        a, e, r = actual_vs_expected(d, at, hz, regime, 3, te)
        mean_h = [float((at * hz)[te & (regime == g)].sum() / at[te & (regime == g)].sum())
                  for g in range(3)]
        rows[label] = (r, mean_h)
    print(f"the TTC macro input is the training-window mean stress, {train_mean:+.2f} points\n")
    print(f"{'window':<22} {'PIT hazard':>11} {'TTC hazard':>11} {'PIT A/E':>9} {'TTC A/E':>9}")
    for g in range(3):
        print(f"{names[g]:<22} {100 * rows['PIT'][1][g]:>10.2f}% {100 * rows['TTC'][1][g]:>10.2f}% "
              f"{rows['PIT'][0][g]:>9.2f} {rows['TTC'][0][g]:>9.2f}")
    swing_pit = max(rows["PIT"][1]) / min(rows["PIT"][1])
    swing_ttc = max(rows["TTC"][1]) / min(rows["TTC"][1])
    print(f"\nlargest swing in average predicted hazard across windows: PIT {swing_pit:.2f}x, "
          f"TTC {swing_ttc:.2f}x")
    print(f"in the recession window the TTC ratio is {rows['TTC'][0][1]:.2f} against the PIT model's "
          f"{rows['PIT'][0][1]:.2f}: the TTC hazard moves only with the age mix of the cells.")
    print("neither is wrong. They answer different questions, and the mistake is to use one "
          "where the other was asked for.")


_try("PIT versus TTC", _show_pit_vs_ttc, needs=_FOR_BACKTEST)

## 12. The flattering numbers, measured

Four protocols that can make a back-test look better than it should. Each is easy to adopt
and hard to see in the final number, and each, run below on this book, prints its score
beside the out-of-time one. The score is the portfolio A/E over the test vintages (1.00 is
perfect) and the spread across vintages (0.00 is perfect).

1. **Random split.** Hold out a random 30% of loans instead of later vintages.
2. **Future in training.** Keep the vintage split, but train on every month of the training
   vintages, including the months after the model would have been built.
3. **Oracle macro.** The first row already scores the model with the macro that actually
   happened: a fair conditional back-test, and a flattering one if it is called a forecast.
   Row 3 scores the same model with the macro frozen at the cut, all a deployed model had.
4. **An outcome-derived feature.** Give each cell its vintage's overall default rate, taken
   from the whole observed life, including the months being predicted.

The first, second and fourth change what the model was *shown*. The third changes what it is
*given at scoring time*. All four are decided before the model runs, by how the data were cut.

In [ ]:
def _rate_feature() -> np.ndarray:
    """Standardised log default rate of the cell's vintage over its WHOLE observed life."""
    at, d = S["at"], S["d"]
    rate = d.sum(axis=1) / at.sum(axis=1)
    z = np.log(rate)
    z = (z - z.mean()) / z.std()
    return np.repeat(z[:, None], MAX_AGE, axis=1)


def _show_flattery() -> None:
    at, d, usable = S["at"], S["d"], OBSERVED & (S["at"] > 0)
    te = S["test"]
    rows = [("out of time, realised macro (conditional)", S["bt_honest"])]

    # 1. random split of loans, model form unchanged
    rng = np.random.default_rng(SEED)
    is_train = rng.random(ORIG.size) < 0.7
    at_a, d_a = risk_table(ORIG[is_train], EXIT_AGE[is_train], EVENT[is_train], N_VINTAGES, MAX_AGE)
    at_b, d_b = risk_table(ORIG[~is_train], EXIT_AGE[~is_train], EVENT[~is_train], N_VINTAGES, MAX_AGE)
    fit_mask = OBSERVED & (np.asarray(at_a) > 0)
    coef_r = fit_on(np.asarray(at_a), np.asarray(d_a), fit_mask)[0]
    hold = OBSERVED & (np.asarray(at_b) > 0) & (CAL_GRID_V >= V_OOT)
    *_, port, spread = backtest(coef_r, np.asarray(at_b), np.asarray(d_b), hold)
    rows.append(("1. random 70/30 split of loans", (port, spread)))

    # 2. future in training: every month of the training vintages
    coef_f = fit_on(at, d, usable & (CAL_GRID_V < V_OOT))[0]
    *_, port, spread = backtest(coef_f, at, d, te)
    rows.append(("2. training months after the cut", (port, spread)))

    # 3. the deployed model knew the macro only up to the cut
    as_of = np.where(np.arange(T_END + 1) <= C_CUT, STRESS, STRESS[C_CUT])
    *_, port, spread = backtest(S["coef_oot"], at, d, te, stress_by_month=as_of)
    rows.append(("3. macro frozen at the cut (as deployed)", (port, spread)))

    # 4. a feature built from the outcomes
    extra = _rate_feature()
    coef_l = fit_on(at, d, S["train"], extra=extra)[0]
    *_, port, spread = backtest(coef_l, at, d, te, extra=extra)
    rows.append(("4. vintage default rate as a feature", (port, spread)))

    S["flattery"] = rows
    print(f"{'protocol':<42} {'portfolio A/E':>14} {'spread':>8}")
    for name, (port, spread) in rows:
        print(f"{name:<42} {port:>14.2f} {spread:>8.2f}")
    honest = rows[0][1]
    cuts = [rows[i] for i in (1, 2, 4)]                   # the three cuts of the evidence
    flatterers = [name.split(".")[0] for name, (p, s) in cuts
                  if abs(np.log(p)) < abs(np.log(honest[0])) and s < honest[1]]
    print(f"\ncuts of the evidence (rows 1, 2 and 4) that beat the out-of-time row on BOTH "
          f"numbers here: {len(flatterers)} of 3" + (f", rows {', '.join(flatterers)}" if flatterers
                                                      else ""))
    print("row 3 is not a cut that flatters. It is the honest model, scored with the inputs it "
          "would really have had.")


_try("flattering protocols", _show_flattery, needs=_FOR_BACKTEST)

Read the table from the top. The random split's training loans include later, worse
vintages and the recession's months, so the model can absorb the drift it was supposed to
fail to see; the out-of-time row cannot. Row 4 is the worst kind of
flattery, because the feature cannot exist when it matters: a vintage originated this month
has no default rate yet. Row 3 is the one that stings. It is not a leak in the model at all;
it is a leak in the *claim*. A PD that needs next year's unemployment to be right is a PD
conditional on a scenario, and the honest name for the back-test is a conditional one.

Every number in that table comes from ONE book. The generator is in this notebook, so how
much of the table belongs to this particular draw can be measured rather than argued: draw
six more books from the same generator with other seeds, and run the out-of-time back-test
and two of the cuts on each. The seeds are the integers 1 to 6, fixed before anything was
measured. Watch which columns move from book to book and which do not.

In [ ]:
OTHER_SEEDS = (1, 2, 3, 4, 5, 6)


def _one_book(seed: int) -> tuple:
    """Quiet-window macro slope, three (A/E, spread) scores and the drift signal of one book."""
    bk = simulate_book(seed)
    tables = lambda keep: [np.asarray(x) for x in risk_table(      # noqa: E731
        bk.orig[keep], bk.exit_age[keep], bk.event[keep], N_VINTAGES, MAX_AGE)]
    at, d = tables(np.ones(bk.orig.size, dtype=bool))
    usable = OBSERVED & (at > 0)
    tr, te = (np.asarray(m) & usable for m in oot_masks(CAL_GRID_V, CAL, V_OOT, C_CUT))

    def fit(a_: np.ndarray, d_: np.ndarray, m: np.ndarray) -> np.ndarray:
        X = design_matrix(CAL_GRID_A[m], bk.stress[CAL[m]], MAX_AGE)
        return np.asarray(fit_newton(X, a_[m].astype(float), d_[m].astype(float))[0])

    coef = fit(at, d, tr)
    a, e, r, port, spread = backtest(coef, at, d, te, stress_by_month=bk.stress)
    pick = np.random.default_rng(seed).random(bk.orig.size) < 0.7
    (at_a, d_a), (at_b, d_b) = tables(pick), tables(~pick)
    rnd = backtest(fit(at_a, d_a, OBSERVED & (at_a > 0)), at_b, d_b,
                   OBSERVED & (at_b > 0) & (CAL_GRID_V >= V_OOT), stress_by_month=bk.stress)[3:]
    fut = backtest(fit(at, d, usable & (CAL_GRID_V < V_OOT)), at, d, te,
                   stress_by_month=bk.stress)[3:]
    tv = np.arange(V_OOT, N_VINTAGES)
    ok = np.isfinite(r[tv]) & (a[tv] > 0)
    corr = float(np.corrcoef(bk.gamma[tv][ok], np.log(r[tv][ok]))[0, 1])
    return float(coef[-1]), (port, spread), rnd, fut, corr


def _show_other_books() -> None:
    rows = [(s, *_one_book(s)) for s in (SEED,) + OTHER_SEEDS]
    print(f"{'book (seed)':>12} {'quiet slope':>12} {'OOT A/E':>8} {'spread':>7} "
          f"{'random A/E':>11} {'spread':>7} {'after-cut A/E':>14} {'spread':>7} {'drift corr':>11}")
    for s, slope, (p, sp), (pr, sr), (pf, sf), c in rows:
        print(f"{s:>12} {slope:>12.3f} {p:>8.2f} {sp:>7.2f} {pr:>11.2f} {sr:>7.2f} {pf:>14.2f} "
              f"{sf:>7.2f} {c:>11.2f}")
    n = len(rows)
    oot = np.array([p for _, _, (p, _), *_ in rows])
    closer = sum(abs(np.log(pr)) < abs(np.log(p)) for _, _, (p, _), (pr, _), *_ in rows)
    beat = sum(abs(np.log(pf)) < abs(np.log(p)) and sf < sp
               for _, _, (p, sp), _, (pf, sf), _ in rows)
    slopes = np.array([r[1] for r in rows])
    link = float(np.corrcoef(slopes, np.log(oot))[0, 1])
    drift = min(r[-1] for r in rows)
    S["other_books"] = (float(oot.min()), float(oot.max()), n)
    print(f"\nout-of-time portfolio A/E across the {n} books: {oot.min():.2f} to {oot.max():.2f}")
    print(f"random split closer to 1.00 than the out-of-time row: {closer} of {n} books")
    print(f"months after the cut in training beat the out-of-time row on both numbers: "
          f"{beat} of {n} books")
    print(f"correlation between the quiet-window slope and log(OOT A/E): {link:+.2f}; "
          f"weakest drift correlation: {drift:.2f}")
    if link < -0.5:
        print("\nthe steeper the slope a book's quiet window happened to give, the lower its "
              "portfolio A/E: a slope that over-predicts")
        print("the recession cancels part of the vintage drift in the total. A portfolio ratio "
              "near 1.00 can be two errors cancelling.")
    if drift > 0.5:
        print("the by-vintage drift correlation stays high in every book: that is the signal "
              "to report, not the portfolio ratio of one draw.")


_try("other books", _show_other_books, needs=_FOR_BACKTEST)

## 13. Exercise 9 — `term_structure()`

Accounting standards ask for losses over a loan's remaining life, so the fitted hazard has to
be turned into a **PD term structure**: the cumulative probability of default by each month
ahead, for a loan made today, under a stated macro scenario. Two treatments matter, and the
standards differ on which applies to which loan (as read on 14 October 2026, sources in
`claims.yaml`): IFRS 9 recognises 12-month expected credit losses on a loan whose credit risk
has not increased significantly since initial recognition, and **lifetime** expected credit
losses on one whose has; US GAAP's CECL measures the allowance over the whole contractual
term. The PD behind either one is a point on the curve you are about to build: the 12-month
PD is its value at month 12, the lifetime PD its value at the end of the term.

The macro overlay is the whole trick. For a loan made in calendar month `o`, its month `k`
is calendar month `o + k`, so the scenario path is read at `o + k`, **not at `k`**. Reading
the path by age is an easy bug to write, and it is silent: the curve still
looks like a curve. A scenario path too short to cover the loan's life is an error, not
something to pad with the last value.

<details><summary>💡 Hint 1 — what to think about</summary>

For a loan originated in month 66, which calendar month is its 4th month on book? Which
entries of the scenario array do you need, and which coefficients go with them? What must
be true of the array's length for that to be possible at all?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Check that the horizon fits the number of age intercepts and that the path reaches the last
month you need. Form the hazard of each month ahead from its own age intercept and the
stress of its own calendar month. The cumulative PD is one minus the running product of the
survival factors, which cannot decrease.
</details>

In [ ]:
def term_structure(coef: np.ndarray, macro_path: np.ndarray, origination: int,
                   horizon: int) -> np.ndarray:
    """Cumulative PD by month ahead, for a loan made in calendar month `origination`.

    `coef` is a fitted vector: `coef[a - 1]` is the intercept of age `a` and `coef[-1]` the
    macro slope. `macro_path[t]` is the stress of calendar month `t`. Month `k = 1..horizon`
    of the loan uses `hazard_k = sigmoid(coef[k - 1] + coef[-1] * macro_path[origination + k])`
    and the result is `1 - cumprod(1 - hazard)`.

    Raises ValueError if `horizon` is below 1 or above the number of ages in `coef`, if
    `origination` is negative, or if `macro_path` does not reach calendar month
    `origination + horizon` (no padding, no wrapping round).

    Example (a flat 10% monthly hazard, macro slope zero, so the path is irrelevant):
        >>> coef = np.array([np.log(1 / 9)] * 3 + [0.0])
        >>> np.round(term_structure(coef, np.zeros(10), origination=2, horizon=3), 3).tolist()
        [0.1, 0.19, 0.271]

    Returns:
        A float array of length `horizon`, non-decreasing and inside [0, 1].
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_term_structure() -> None:
    coef = np.array([np.log(1 / 9)] * 3 + [0.0])
    got = np.asarray(term_structure(coef, np.zeros(10), origination=2, horizon=3))
    assert got.shape == (3,) and np.allclose(got, [0.1, 0.19, 0.271]), (
        f"a flat 10% hazard gives cumulative PDs [0.1, 0.19, 0.271]; got {np.round(got, 4).tolist()}. "
        "Cumulate as 1 - product(1 - h); summing the hazards gives 0.3 at month 3, and "
        "returning the hazards themselves is not a term structure")
    coef2 = np.array([-2.0, -2.0, -2.0, 1.0])
    path = np.array([0.0, 0.0, 0.0, 0.0, 3.0, 0.0, 0.0, 0.0])   # a spike at calendar month 4
    low = np.asarray(term_structure(coef2, path, origination=1, horizon=3))
    at_age = np.asarray(term_structure(coef2, np.array([0.0, 3.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]),
                                       origination=1, horizon=3))
    assert low[2] - low[1] > at_age[2] - at_age[1] + 0.05, (
        "the spike at calendar month 4 belongs to a loan made in month 1 at age 3, so month 3 "
        "must carry the big hazard. Reading the macro path by age instead of by calendar "
        "month puts it in the wrong month of the wrong loan")
    assert (np.diff(low) >= 0).all(), "a cumulative PD can never fall as the horizon grows"
    for bad in ((coef, np.zeros(4), 2, 3), (coef, np.zeros(10), 2, 4)):
        try:
            term_structure(*bad)
        except ValueError:
            continue
        raise AssertionError("a path too short, or a horizon beyond the fitted ages, must "
                             "raise ValueError rather than pad or clip")
    print("exercise 9 looks right — the scenario is read by calendar month, not by age")

In [ ]:
_try("exercise 9", _check_term_structure)

## 14. Exercise 10 — `expected_credit_loss()`

A PD is not a loss. **Expected credit loss** (ECL) prices it: the probability of defaulting in
each month, times the exposure outstanding then, times the loss given default (LGD, the share
of that exposure not recovered). The month in question is the month of default, so the
weight is the **marginal** PD, the probability of first defaulting in exactly month `k`,
which is the *increase* in the cumulative curve, not the curve itself.

Sum the marginal terms over 12 months and you have IFRS 9's stage-1 measure; sum them over
the whole remaining term and you have a lifetime measure, which is what CECL and IFRS 9's
later stages use. Discounting is left out to keep the lesson about PD, and so is
prepayment: US supervisory guidance on CECL says expected prepayments are considered when
the contractual term is determined, but the curve below is the no-prepayment one and the
exposure a plain straight-line balance, so the lifetime figures overstate what a measurement
with expected prepayments would give. The model card lists it as a known gap.

<details><summary>💡 Hint 1 — what to think about</summary>

From a cumulative curve `[0.1, 0.19, 0.271]`, what is the probability of defaulting
specifically in month 3? What is the marginal PD of month 1, where there is no earlier
value to subtract? What should a horizon of 12 do when only 8 months remain?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Difference the cumulative curve after putting a zero in front of it, keep the first
`horizon` terms (or all of them for `None`, and never more than exist), weight each by its
month's exposure, multiply by the LGD and add up. Reject an LGD outside `[0, 1]`, a
horizon below 1, or mismatched lengths with `ValueError`.
</details>

In [ ]:
def expected_credit_loss(cum_pd: np.ndarray, exposure: np.ndarray, lgd: float,
                         horizon: "int | None" = None) -> float:
    """Undiscounted expected credit loss over the first `horizon` months, or the whole term.

    `cum_pd[k - 1]` is the cumulative PD by month `k` and `exposure[k - 1]` the exposure at
    default in month `k`, as a share of the original balance. With marginal PD
    `m_k = cum_pd[k - 1] - cum_pd[k - 2]` (and `m_1 = cum_pd[0]`):
        ECL = lgd * sum_{k <= horizon} exposure[k - 1] * m_k.
    `horizon=None` means every month; a horizon beyond the curve means the whole curve.

    Raises ValueError if `lgd` is outside [0, 1], `horizon` is below 1, or the two arrays are
    not 1-D and of equal length.

    Example (three months; LGD one half):
        >>> cum, expo = np.array([0.1, 0.19, 0.271]), np.array([1.0, 0.5, 0.25])
        >>> round(expected_credit_loss(cum, expo, 0.5, horizon=2), 5)
        0.0725
        >>> round(expected_credit_loss(cum, expo, 0.5), 6)
        0.082625

    Returns:
        A Python float.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_expected_credit_loss() -> None:
    cum, expo = np.array([0.1, 0.19, 0.271]), np.array([1.0, 0.5, 0.25])
    short = expected_credit_loss(cum, expo, 0.5, horizon=2)
    full = expected_credit_loss(cum, expo, 0.5)
    assert type(short) is float and type(full) is float, "return a plain Python float"
    assert abs(short - 0.0725) < 1e-9, (
        f"12-month style ECL over two months should be 0.5 * (1.0 * 0.10 + 0.5 * 0.09) = "
        f"0.0725, got {short:.6f}. Weight each month by its MARGINAL PD (0.10, 0.09), the "
        "increase in the cumulative curve, not by the cumulative values")
    assert abs(full - 0.082625) < 1e-9, (
        f"the lifetime ECL over all three months should be 0.082625, got {full:.6f}")
    assert abs(expected_credit_loss(cum, expo, 0.5, horizon=12) - full) < 1e-12, (
        "a horizon longer than the remaining term means the whole term, not an error")
    try:
        expected_credit_loss(cum, expo, 1.5)
    except ValueError:
        print("exercise 10 looks right — ECL is marginal PD x exposure x LGD, summed over a horizon")
    else:
        raise AssertionError("an LGD above 1 must raise ValueError")

In [ ]:
_try("exercise 10", _check_expected_credit_loss)

Now the overlay on the real book. The final model is refitted on **all** observed cells:
the recession is now in its data, so the macro slope has something to learn from. The next
loan to be made is a vintage-66 loan, and two scenarios describe the economy it will live
through: a **baseline**, in which the current stress fades with a half-life of a year, and an
**adverse** path, in which a second downturn of the same shape as the first arrives.

In [ ]:
_FOR_OVERLAY = _FOR_SPLIT + ("exercise 4", "exercise 5", "exercise 7", "exercise 9")
ORIGINATION = T_END + 1
LGD = 0.60          # an assumption for the demonstration, not an estimate from this book


def scenario_path(kind: str, length: int) -> np.ndarray:
    """Stress by calendar month: the observed path, then a scenario. Given to you."""
    t = np.arange(T_END + 1, length)
    fade = STRESS[T_END] * 0.5 ** ((t - T_END) / 12.0)
    if kind == "baseline":
        future = fade
    elif kind == "adverse":
        x = np.clip(t - (T_END + 2), 0.0, None)
        future = fade + 5.0 * (1.0 - np.exp(-x / 3.0)) * np.exp(-x / 14.0)
    else:
        raise ValueError(kind)
    return np.concatenate([STRESS, future])


def _show_scenarios() -> None:
    at, d = S["at"], S["d"]
    coef = fit_on(at, d, OBSERVED & (at > 0))[0]
    S["coef_all"] = coef
    length = ORIGINATION + MAX_AGE + 1
    paths = {k: scenario_path(k, length) for k in ("baseline", "adverse")}
    curves = {k: np.asarray(term_structure(coef, p, ORIGINATION, MAX_AGE)) for k, p in paths.items()}
    S["scenario_curves"] = curves
    print(f"final model refitted on {int((OBSERVED & (at > 0)).sum())} cells; macro slope "
          f"{coef[-1]:.3f} (planted {BETA_TRUE:.3f})\n")
    print(f"{'PD of a vintage-' + str(ORIGINATION) + ' loan':<30} {'baseline':>10} {'adverse':>10} {'adverse / baseline':>20}")
    for label, k in (("12-month PD", 12), ("24-month PD", 24), (f"lifetime PD ({MAX_AGE} months)", MAX_AGE)):
        b, a = curves["baseline"][k - 1], curves["adverse"][k - 1]
        print(f"{label:<30} {100 * b:>9.1f}% {100 * a:>9.1f}% {a / b:>20.2f}")
    ok = all(bool((np.diff(c) >= 0).all()) for c in curves.values())
    print(f"\nboth curves non-decreasing in the horizon: {ok}")
    print(f"the adverse curve is at or above the baseline at every horizon: "
          f"{bool((curves['adverse'] >= curves['baseline'] - 1e-12).all())}")
    fig, ax = plt.subplots(figsize=(8, 3))
    for k, c in curves.items():
        ax.plot(np.arange(1, MAX_AGE + 1), 100 * c, label=k)
    ax.set_xlabel("months ahead")
    ax.set_ylabel("cumulative PD, %")
    ax.legend(fontsize=8)
    fig.tight_layout()
    _show(fig)


_try("scenario term structures", _show_scenarios, needs=_FOR_OVERLAY)

In [ ]:
def _show_ecl() -> None:
    curves = S["scenario_curves"]
    exposure = 1.0 - np.arange(MAX_AGE) / MAX_AGE       # a straight-line amortising balance
    print(f"expected credit loss per 1.00 of original balance, LGD {LGD:.2f}, undiscounted\n")
    print(f"{'':<12} {'12-month':>10} {'lifetime':>10} {'lifetime / 12-month':>21}")
    for k in ("baseline", "adverse"):
        e12 = expected_credit_loss(curves[k], exposure, LGD, horizon=12)
        elife = expected_credit_loss(curves[k], exposure, LGD)
        print(f"{k:<12} {e12:>10.4f} {elife:>10.4f} {elife / e12:>21.2f}")
    b = expected_credit_loss(curves["baseline"], exposure, LGD, horizon=12)
    a = expected_credit_loss(curves["adverse"], exposure, LGD, horizon=12)
    print(f"\nthe adverse scenario moves the 12-month ECL by {a / b:.2f}x.")
    print("a loan that moves from stage 1 to stage 2 changes measure, from the first column to "
          "the second, with no change in the model.")
    print("that step is a cliff in the provision, which is why where the line for a "
          "significant increase in credit risk is drawn matters.")


_try("expected credit loss", _show_ecl, needs=_FOR_OVERLAY + ("exercise 10",))

## 15. Common mistakes

- **Dropping censored loans, or scoring them as good.** They are at risk for every month
  they were watched and are defaults in none. Dropping them inflates later hazards; padding
  them as good deflates the recent vintages and makes them look well underwritten.
- **Carrying a vintage curve past the data.** A flat tail is not a forecast. It is a
  month nobody lived, drawn as a month with no defaults.
- **Dividing by the loans originated instead of the loans at risk.** A prepaid loan is
  scored as a survivor for ever, so every hazard after its exit is diluted and the curve
  reads low.
- **Splitting loans at random.** The test loans have training siblings in the same vintage
  and the same calendar months. It measures interpolation, and lending is extrapolation.
- **Training on months after the model's date, or scoring with next year's macro.** Both
  put the future into a number that will be presented as a forecast.
- **One intercept and an age slope.** Default risk has a hump in months on book. The line
  cannot draw it, and the misfit lands in whatever else the model has, usually the macro
  slope.
- **Reading the scenario by age.** The macro path is indexed by calendar month. The bug is
  silent: the curve still looks like a curve.
- **Calling a PD "through the cycle" because it ignores the macro.** A TTC PD needs a
  window that contains a cycle. Fitted on a quiet stretch, it is a quiet-stretch PD.
- **Trusting a gradient because the fit ran.** A wrong gradient converges to something.
  Check it against finite differences before you believe a coefficient.

The vintage effect is worth one more look before the self-check. The model has no term for it,
and the by-vintage table showed what that costs. Run this.

In [ ]:
def _show_vintage_effect() -> None:
    a, e, r, port, spread = backtest(S["coef_oot"], S["at"], S["d"], S["test"])
    test_v = np.arange(V_OOT, N_VINTAGES)
    ok = np.isfinite(r[test_v])
    corr = float(np.corrcoef(BOOK.gamma[test_v][ok], np.log(r[test_v][ok]))[0, 1])
    print(f"correlation between the planted vintage quality and log(A / E), test vintages: {corr:.2f}")
    if corr > 0.5:
        print("the by-vintage back-test recovered a hidden variable the model was never given, "
              "which is what it is for.")
    at, d = S["at"], S["d"]
    slope = float(fit_on(at, d, OBSERVED & (at > 0))[0][-1])
    print(f"\nrefitted on every observed cell, the model reads a macro slope of {slope:.3f} "
          f"against the planted {BETA_TRUE:.3f}.")
    if slope > BETA_TRUE:
        print("the late, worse vintages met the recession, and a model with no vintage term "
              "hands their extra defaults to the macro:")
        print("the scenario PDs of section 14 inherit that steeper slope. That is confounding, "
              "and the by-vintage table is how you see it.")
    print("the fix is not a vintage dummy (a new vintage has no dummy to fit) but a feature "
          "known at origination that moves with quality, a later module's subject, or an "
          "overlay that says so out loud.")


_try("vintage effect", _show_vintage_effect, needs=_FOR_BACKTEST)

## 16. Self-check

1. A vintage originated eight months before the data cut-off shows a lower cumulative
   default rate at month 24 than an older vintage. The right reading is:
   - (a) it was underwritten better, because it has fewer defaults
   - (b) month 24 of the young vintage was never observed, so any number printed there
         counts unobserved months as clean, and there is nothing to compare
   - (c) the older vintage is inflated by prepayments

2. A loan prepays in its ninth month. In the risk table it should be:
   - (a) counted as a default in month nine, because the loan left
   - (b) removed from the table entirely, as though it had never been made
   - (c) at risk in months one to nine with no default, and absent from month ten on

3. A PD model trained on a random 70% of loans posts a portfolio actual-over-expected close
   to perfect on the other 30%. The same model trained on early vintages misses on later
   ones. The best explanation is:
   - (a) the random split shows the model loans from the same vintages and calendar months
         it will be scored on, including later, worse ones, so it has been shown its future
   - (b) the random split simply has more training data
   - (c) later vintages are harder to predict because they are younger

4. A PD that rises with the unemployment rate and falls when it falls is:
   - (a) through-the-cycle, because it is tied to the cycle
   - (b) point-in-time, and a lifetime PD from it needs a stated macro scenario for the
         months ahead
   - (c) unusable for lifetime expected credit loss, because the future is unknown

5. A colleague adds each vintage's overall default rate, computed over its whole observed
   life, as a feature, and the back-test improves sharply. This is:
   - (a) a vintage effect, and a good one to have
   - (b) fine, because the rate is known by the end of the month
   - (c) target leakage: the feature is built from the outcomes being predicted, and does
         not exist for a vintage made this month

Answers, with the reasoning, come with this lesson's worked solution when you enrol on Synapsa.

One last cell, and it is the deliverable. A PD without its scenario is a number somebody will
later quote out of context. Print the model card line.

In [ ]:
def _model_card() -> None:
    curves = S["scenario_curves"]
    print("PD MODEL CARD — hand this on with every line together")
    print(f"  model        discrete-time hazard, one intercept per month on book "
          f"({MAX_AGE}), one macro slope, fitted by Newton's method")
    print(f"  data         synthetic book, {N_VINTAGES} vintages x {LOANS_PER_VINTAGE:,} loans, "
          f"months 0..{T_END}; censoring handled by risk-set counts")
    print(f"  validation   out of time: built at month {C_CUT} on vintages < {V_OOT}, tested on "
          f"vintages >= {V_OOT}; portfolio A/E {S['bt_honest'][0]:.2f}, "
          f"spread {S['bt_honest'][1]:.2f}, with the realised macro")
    if "other_books" in S:
        lo, hi, n = S["other_books"]
        print(f"               (one draw: {lo:.2f} to {hi:.2f} across {n} books from the same "
              "generator)")
    print(f"  scenarios    vintage-{ORIGINATION} lifetime PD: baseline "
          f"{100 * curves['baseline'][-1]:.1f}%, adverse {100 * curves['adverse'][-1]:.1f}%")
    print("  known gaps   no vintage-quality term; TTC average taken over a window with no "
          "recession; LGD and exposure are assumptions;")
    print("               the lifetime ECL ignores prepayment; the out-of-time figures above are "
          "one draw of the generator")


_try("model card", _model_card, needs=_FOR_OVERLAY + ("exercise 8",))

## What you built, and where it goes next

You now have the machinery a lifetime-PD model is assembled from: a risk table that respects
censoring, vintage curves that stop where the data stop, a hazard model fitted by an
optimiser you checked yourself, a macro overlay read by calendar month, and a back-test
protocol whose score you can trust because it was cut before the model ran. The remaining
modules of this programme take a cut of the same problem each: the loans you never saw
because you declined them, the choice between a scorecard and boosting, the reasons you owe a
declined applicant, disparity you must test without the attribute, thin files, and
the monitoring file that a validator will ask for.

The habit to carry forward: **the honest number is decided before the model runs.** How the
data were cut, which months the model was allowed to see and which macro it is given at
scoring time are choices about the evidence, and no amount of modelling after the fact
repairs a cut that flattered.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_risk_table),
                              ("exercise 2", _check_vintage_curves),
                              ("exercise 3", _check_oot_masks),
                              ("exercise 4", _check_design_matrix),
                              ("exercise 5", _check_binomial_nll_grad),
                              ("exercise 6", _check_gradient_check),
                              ("exercise 7", _check_fit_newton),
                              ("exercise 8", _check_actual_vs_expected),
                              ("exercise 9", _check_term_structure),
                              ("exercise 10", _check_expected_credit_loss)):
            _try(_name, _check)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.